# 01c — EffNet-B0 Random-Image Baseline (20 Seeds)

**Measuring Leaky Variance & 20 vs 20 Welch Grouping Test**

Hypothesis Test:
- Grouping Arm: 20 seeds of `effnet_b0` on `block_shuffled` (mean=88.59%, SD=5.93 pp).
- Leaky Baseline Arm: 20 seeds of `effnet_b0` on `random_image`.
- Welch t-test & 10k bootstrap CI on the difference in means.
- Halt condition: `random_image` SD > 4.0 pp.


In [ ]:
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "GPU accelerator required!"


In [ ]:
SEEDS = [42, 123, 2024, 7, 2718, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16]
MODEL = "effnet_b0"
STRAT = "random_image"

OUT = f"{RESULTS}/01c_random_image_null"
os.makedirs(OUT, exist_ok=True)

done = {os.path.basename(f).replace("_result.json", "")
        for f in os.listdir(OUT) if f.endswith("_result.json")} if os.path.isdir(OUT) else set()
print("already complete:", len(done), "target:", len(SEEDS))


In [ ]:
results = []
t_start = time.time()
for seed in SEEDS:
    name = f"{MODEL}__{STRAT}__seed{seed}"
    if name in done:
        print("skip", name)
        continue
    r = engine.run_experiment(ROOT, MODEL, STRAT, seed, OUT)
    results.append(r)
    print(f"    elapsed {(time.time()-t_start)/60:.2f} min")


In [ ]:
import numpy as np
import statistics as st
from scipy import stats

all_res = engine.collect_results(OUT)
accs = [r["accuracy"] for r in all_res]
ben_f1s = [r["benign_f1"] for r in all_res]
seeds_run = [r["seed"] for r in all_res]

# Reference block_shuffled 20-seed accuracies
shuffled_accs = [
    97.436, 87.179, 95.763, 86.555, 88.983, 90.517, 93.043, 84.348, 84.746, 95.69,
    83.898, 94.118, 88.136, 92.241, 75.214, 76.923, 86.957, 94.915, 87.931, 87.179
]

print("=" * 60)
print(f"EFFNET_B0 RANDOM_IMAGE BASELINE (n={len(accs)})")
print("=" * 60)
mean_rand = float(st.mean(accs))
sd_rand = float(st.stdev(accs)) if len(accs) > 1 else 0.0
median_rand = float(np.median(accs))
iqr_rand = float(np.percentile(accs, 75) - np.percentile(accs, 25))
ci95_rand = [float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))]

print(f"Mean Accuracy:    {mean_rand:.2f}% +/- {sd_rand:.2f}%")
print(f"Median Accuracy:  {median_rand:.2f}% (IQR: {iqr_rand:.2f} pp)")
print(f"Central 95% CI:   [{ci95_rand[0]:.2f}%, {ci95_rand[1]:.2f}%]")
print(f"Mean Benign F1:   {st.mean(ben_f1s):.2f}% +/- {st.stdev(ben_f1s) if len(ben_f1s) > 1 else 0:.2f}%")

# Welch t-test (20 vs 20)
t_stat, p_val = stats.ttest_ind(shuffled_accs, accs, equal_var=False)
mean_shuff = float(np.mean(shuffled_accs))
grouping_drop = mean_rand - mean_shuff

# Bootstrap 95% CI on difference in means (10,000 resamples)
rng = np.random.default_rng(42)
B = 10000
boot_diffs = np.zeros(B)
for b in range(B):
    s_shuff = rng.choice(shuffled_accs, size=len(shuffled_accs), replace=True)
    s_rand = rng.choice(accs, size=len(accs), replace=True)
    boot_diffs[b] = s_rand.mean() - s_shuff.mean()

ci_diff_lo = float(np.percentile(boot_diffs, 2.5))
ci_diff_hi = float(np.percentile(boot_diffs, 97.5))

print("-" * 60)
print(f"WELCH 20 vs 20 COMPARISON (random_image vs block_shuffled):")
print(f"  Grouping Drop (mean diff): {grouping_drop:.2f} pp")
print(f"  Bootstrap 95% CI:          [{ci_diff_lo:.2f}, {ci_diff_hi:.2f}] pp")
print(f"  Welch t-statistic:         {t_stat:.3f}")
print(f"  Welch p-value:             {p_val:.4e}")
print(f"  Difference excludes zero:  {ci_diff_lo > 0}")

halt_sd_high = sd_rand > 4.0
if halt_sd_high:
    print(f"-> HALT CONDITION TRIGGERED: random_image SD is {sd_rand:.2f} pp (> 4.0 pp)!")

summary = {
    "model": MODEL,
    "strategy": STRAT,
    "n_seeds": len(accs),
    "seeds": seeds_run,
    "accuracies": [round(float(a), 3) for a in accs],
    "mean_acc": round(mean_rand, 3),
    "sd_acc": round(sd_rand, 3),
    "median_acc": round(median_rand, 3),
    "iqr_acc": round(iqr_rand, 3),
    "ci95_acc": [round(ci95_rand[0], 3), round(ci95_rand[1], 3)],
    "welch_grouping_drop_pp": round(grouping_drop, 3),
    "welch_p_value": float(p_val),
    "welch_t_stat": round(float(t_stat), 3),
    "bootstrap_ci95_drop": [round(ci_diff_lo, 3), round(ci_diff_hi, 3)],
    "halt_sd_high": bool(halt_sd_high)
}

with open(f"{OUT}/random_image_null_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("=" * 60)
